<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Hand-made features

**[Hand-made features](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/)** · Vision AI: See, Count and Inspect · Module 2, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** describe the shapes in a photo with HOG features, train a linear model on them, compare it with the colour histogram, and combine the two.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Colour histograms and nearest neighbours, from [Rules and colour histograms](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M02-L02-hand-made-features/hand-made-features-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 15 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The photos and the two feature functions

`hog_features` makes a photo grey and small (128 × 128), finds the edges, and counts their directions in each cell of 16 × 16 pixels. `colour_histogram` is the function of the last lesson.

In [ ]:
from skimage.feature import hog

def hog_features(path, size=128):
    """HOG: the edge directions in each 16 x 16 cell of a grey 128 x 128 photo."""
    g = np.asarray(Image.open(path).convert("L").resize((size, size)), float) / 255
    return hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)

def colour_histogram(path, bins=(8, 4, 4)):
    """The colour histogram of the last lesson: 8 x 4 x 4 HSV bins, square root."""
    hsv = np.asarray(Image.open(path).convert("RGB").resize((128, 128)).convert("HSV")).reshape(-1, 3)
    h, _ = np.histogramdd(hsv, bins=bins, range=((0, 256), (0, 256), (0, 256)))
    return np.sqrt(h.ravel() / h.sum())

catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
studio = pd.read_csv("data/catalogue/new_studio_manifest.csv")
f = hog_features("data/" + train["file"][0])
print(f.shape)

> **Check.** You should see `(1764,)`. The photo has 8 × 8 cells. HOG groups them in blocks of 2 × 2 cells: 7 × 7 = 49 blocks, each with 4 cells × 9 directions = 36 numbers. 49 × 36 = 1764.

## 2. See what HOG sees

`visualize=True` draws the main edge directions of each cell as short lines.

In [ ]:
path = "data/" + test.loc[test["label"] == "chair", "file"].iloc[5]
g = np.asarray(Image.open(path).convert("L").resize((128, 128)), float) / 255
_, picture = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(Image.open(path)); axes[0].set_title("photo")
axes[1].imshow(picture ** 0.6, cmap="gray_r"); axes[1].set_title("HOG")
for ax in axes: ax.axis("off")
plt.show()

## 3. HOG and a linear model

A **logistic regression** gives each of the 1764 numbers a weight per class and adds them up. `StandardScaler` first puts every number on the same scale.

> **Predict.** HOG has no colour. The new studio changes only the light and the colour. What happens to HOG's accuracy on the new studio photos?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import time

start = time.perf_counter()
Htr = np.stack([hog_features("data/" + f) for f in train["file"]])
Hte = np.stack([hog_features("data/" + f) for f in test["file"]])
ms = 1000 * (time.perf_counter() - start) / (len(train) + len(test))
Hst = np.stack([hog_features("data/" + f) for f in studio["file"]])
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000)).fit(Htr, train["label"])
for name, X, y in (("catalogue test", Hte, test["label"]), ("new studio", Hst, studio["label"])):
    s = cp.classification_scores(list(y), list(model.predict(X)), CLASSES)
    print(f"{name:15s} accuracy {s['accuracy']}  worst {s['worst_class']} {s['worst_recall']}")
print(f"HOG features: {ms:.1f} ms per photo")

> **Check.** You should see 0.356 on the test photos and 0.356 on the new studio photos: the warm light does not move HOG at all. The worst class is candle: HOG sees the outline, and a candle's outline changes a lot between photos. Your time per photo will differ from ours (about 1 ms).

## 4. The recall of each class

In [ ]:
s = cp.classification_scores(list(test["label"]), list(model.predict(Hte)), CLASSES)
pd.DataFrame(s["per_class"]).T[["precision", "recall", "n"]]

## 5. Your turn: shape and colour together

HOG knows shapes and nothing about colour. The colour histogram knows colour and nothing about shapes. Put the two side by side in one list of numbers per photo, and train the same model.

> **Your turn.** Make `Xtr_both` and `Xte_both` with `np.hstack` (the HOG numbers first, then the colour histogram), train the same pipeline (`StandardScaler` and `LogisticRegression(C=0.5, max_iter=3000)`) on the training photos, and put the test accuracy in `acc_both` (as `cp.classification_scores` gives it).

In [ ]:
Xtr_both = ...
Xte_both = ...
acc_both = ...

In [ ]:
expect('acc_both', acc_both, 0.487, tolerance=0.002)

## 6. Add your rows to the comparison table

The course helper keeps a table. Add your HOG row, and compare it with the course's rows.

In [ ]:
mine = cp.Table()
mine.add(cp.classification_row("HOG + logistic regression (yours)", list(test["label"]), list(model.predict(Hte)),
                               "catalogue test", round(ms, 2), 0, len(train), "none (scikit-image, BSD)"))
mine.show("catalogue", columns=("method", "setting", "value", "worst", "worst_value", "ms_per_image"))

## 7. What HOG cannot see

HOG is fast and does not care about the light. But it describes edges in fixed cells: a mug turned a little, or photographed from further away, puts its edges in other cells. The model then sees a different list of numbers. Write one sentence: which Larkfield photos would suit HOG better, the catalogue photos or the photos of a fixed camera over the returns table? Why?

## Next

You added a shape method to the comparison table. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Convolution and patches, in one lesson](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.